# Plot

In [ ]:
library(readr)
library(purrr)
library(dplyr)
library(stringr)
library(fuzzyjoin)
library(ggplot2)
library(dplyr)
library(tidyverse)
library(Matrix)
library(reshape2)
library(RColorBrewer)
library(dplyr)
library(rstatix)

options(tibble.width = Inf)

## Plot parameters

In [ ]:
out_dir = "/ceph.groups/mshahbazi.grp/rsakata/Figures/ZVAD_blastoid_BF/output"
filter_dapi = TRUE

analysis_summary_files = c(
"/ceph.groups/mshahbazi.grp/rsakata/Figures/ZVAD_blastoid_BF/Blastoid_counts.csv")

In [ ]:
# define const for visualization
FONT.SIZE <- 7
LABEL.FONT.SIZE <- 7
w <- 2 
h <- 2.5
LINE.W <- 0.5/2.141959

# Set geom defaults globally
update_geom_defaults("line",      list(linewidth = LINE.W))
update_geom_defaults("errorbar",  list(linewidth = LINE.W))
#update_geom_defaults("point",     list(size = LINE.W, stroke = LINE.W))

settheme <- theme_minimal() + 
  theme(
    text = element_text(family = "sans"), 
    panel.background = element_blank(),
    panel.grid.major = element_blank(), 
    panel.grid.minor = element_blank(),
    plot.background = element_blank(),
    axis.ticks = element_line(colour = "black", linewidth = LINE.W),
    axis.ticks.length = unit(0.1, "cm"), 
    axis.line = element_line(linewidth = LINE.W, colour = "black"),
    axis.title = element_text(size = FONT.SIZE),
    axis.text = element_text(colour = "black", size = FONT.SIZE),
    strip.text = element_text(size = FONT.SIZE), 
    strip.text.y.left = element_text(angle = 0, hjust = 1, size = FONT.SIZE),
    legend.position = "right",
    legend.title = element_text(size = FONT.SIZE), 
    legend.text = element_text(size = FONT.SIZE),
    legend.key.size = unit(0.3, "cm"),
    axis.text.x = element_text(colour = "black", angle = 0, size = LABEL.FONT.SIZE),
    title = element_text(size = FONT.SIZE) 
  )

In [ ]:
color_condition = c("Control" = "#7BBA56", 
               "Reversine" = "#87549B", 
               "Mosaic" = "#E8973E")

col_Annexin= "#4db2cb"

col_ZVAD = c("ZVAD+" = "#6a6969", 
               "ZVAD-" = "#bbbbbb")

col_ZVAD2 = c("ZVAD+" = "#022047", 
               "ZVAD-" = "#157AFF")

col_GFP = "#7BBA56"
col_RFP = "#cb377c"

## 1. Extract summary files

In [ ]:
merged_df <- read_csv(analysis_summary_files)


In [ ]:
head(merged_df)

merged_df %>%
  count(timepoint, Population, condition_1, condition_2, condition_3)

In [ ]:
df_sample = merged_df

# Plot 

### A) Proportion of structures 

In [ ]:
title = "per_Annexinpos_D4"
w <- 2.5
h <- 1.6
options(repr.plot.width=w, repr.plot.height=h)
  
p = ggplot(df_sample, aes(x = Condition, y = perDeveloped , group = ZVAD, fill = ZVAD)) +  # dots for each file
    stat_summary(
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75), alpha = 0.6, width = 0.6) +   # error bars
    geom_jitter(
      aes(fill = Condition),
      position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
      size = 0.5, alpha = 0.8, color = "black"
    )  +  # average bar
    stat_summary(
      fun.data = mean_se, 
      geom = "errorbar",
      position = position_dodge(width = 0.75),
      width = 0.2, 
      color = "black")+
    labs(
      title = title,
      y = "proportion developed",
      x = "condition"
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1),
      #legend.position = "none", 
      legend.key.size = unit(0.3, "cm")
      
    ) +
      scale_y_continuous(limits = c(0, 1.10), expand = c(0, 0))+
      #facet_wrap( ~ condition_2) +
      scale_fill_manual(values=col_ZVAD2)

ggsave(file.path(out_dir, sprintf("A_%s.pdf", title)),
                plot = p, width = w, height = h)

p

In [ ]:
title = "per_Annexinpos_D4"
w <- 2.5
h <- 1.8
options(repr.plot.width=w, repr.plot.height=h)
  
p = ggplot(df_sample, aes(x = Condition, y = perDeveloped , group = ZVAD, fill = ZVAD)) +  # dots for each file
    stat_summary(
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75), alpha = 0.6, width = 0.6,
      fill = "grey") +   # error bars
    geom_jitter(
      aes(color = ZVAD),
      position = position_jitterdodge(jitter.width = 0.1, dodge.width = 0.75),
      size = 0.5, alpha = 0.8
    )  +  # average bar
    stat_summary(
      fun.data = mean_se, 
      geom = "errorbar",
      position = position_dodge(width = 0.75),
      width = 0.2, 
      color = "black")+
    labs(
      title = title,
      y = "proportion developed",
      x = "condition"
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1),
      #legend.position = "none", 
      legend.key.size = unit(0.3, "cm")
      
    ) +
      scale_y_continuous(limits = c(0, 1.10), expand = c(0, 0))+
      #facet_wrap( ~ condition_2) +
      scale_color_manual(values=col_ZVAD2)

ggsave(file.path(out_dir, sprintf("A_%s.pdf", title)),
                plot = p, width = w, height = h)

p

## For contingency

In [ ]:
head(df_sample)

In [ ]:
df_exp <- df_sample %>%
  group_by(EXP, Condition, ZVAD) %>%
  summarise(
    developed = sum(Developed),
    failed = sum(Failed),
    proportion_developed = developed/(developed+failed),
    .groups = "drop"
  )

df_exp

In [ ]:
library(dplyr)
library(emmeans)

df_test <-df_exp %>%
  mutate(
    EXP = factor(EXP),
    Condition = factor(
      Condition,
      levels = c("control", "mosaic", "reversine")
    ),
    ZVAD = factor(
      ZVAD,
      levels = c("ZVAD-", "ZVAD+")
    )
  )

# Additive: one common ZVAD effect across Conditions
fit_additive <- glm(
  cbind(developed, failed) ~ EXP + Condition + ZVAD,
  family = binomial, data = df_test
)

# Interaction: ZVAD effect allowed to differ by Condition
fit_interaction <- glm(
  cbind(developed, failed) ~ EXP + Condition * ZVAD,
  family = binomial, data = df_test
)

# test is the result different between with and without inteaction
anova(
  fit_additive,
  fit_interaction,
  test = "LRT"
)

#Wald test
summary(fit_additive)

#LRT test
drop1(fit_additive, test = "LRT") 


In [ ]:
# test between groups
emm <- emmeans(
  fit_interaction,
  ~ ZVAD | Condition
)

zvad_comparisons <- contrast(
  emm,
  method = list(
    "ZVAD+ vs ZVAD-" = c(-1, 1)
  )
)

zvad_results <- summary(
  zvad_comparisons,
  type = "response",
  infer = c(TRUE, TRUE),
  adjust = "holm"
) %>%
  as.data.frame()%>%
  mutate(
    p_use = if ("p.adj" %in% names(.)) p.adj else p.value,   # fall back to raw p
    stars = case_when(
      p_use < 0.0001 ~ "****",
      p_use < 0.001  ~ "***",
      p_use< 0.01   ~ "**",
      p_use < 0.05   ~ "*",
      TRUE           ~ "ns"
    )
  )

zvad_results